CELL 1 — Imports


In [1]:
import pandas as pd
import requests
import time
from datetime import datetime
from tqdm import tqdm
import os

CELL 2 — Create Data Folder Path

In [2]:
DATA_DIR = "../data"

os.makedirs(DATA_DIR, exist_ok=True)

print("Data folder ready:", DATA_DIR)

Data folder ready: ../data


CELL 3 — Binance API Config

We use Binance because:

more stable
huge history
free
easy


In [3]:
SYMBOL = "BTCUSDT"
INTERVAL = "5m"

BASE_URL = "https://api.binance.com/api/v3/klines"

LIMIT = 1000

CELL 4 — Date Range

In [4]:
start_date = "2020-01-01"
end_date = "2026-01-01"

start_ts = int(pd.Timestamp(start_date).timestamp() * 1000)
end_ts = int(pd.Timestamp(end_date).timestamp() * 1000)

print(start_ts, end_ts)

1577836800000 1767225600000


CELL 5 — Download Function

In [5]:
def fetch_klines(symbol, interval, start_ts, end_ts, limit=1000):

    params = {
        "symbol": symbol,
        "interval": interval,
        "startTime": start_ts,
        "endTime": end_ts,
        "limit": limit
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print("Error:", response.text)
        return []

    return response.json()

CELL 6 — Download All Historical Data

In [6]:
all_data = []

current_ts = start_ts

while current_ts < end_ts:

    data = fetch_klines(
        SYMBOL,
        INTERVAL,
        current_ts,
        end_ts,
        LIMIT
    )

    if not data:
        print("No more data.")
        break

    all_data.extend(data)

    last_open_time = data[-1][0]

    current_ts = last_open_time + 1

    print(
        "Downloaded candles:",
        len(all_data),
        "| Last:",
        datetime.fromtimestamp(last_open_time / 1000)
    )

    time.sleep(0.2)

print("Finished.")

Downloaded candles: 1000 | Last: 2020-01-04 16:45:00
Downloaded candles: 2000 | Last: 2020-01-08 04:05:00
Downloaded candles: 3000 | Last: 2020-01-11 15:25:00
Downloaded candles: 4000 | Last: 2020-01-15 02:45:00
Downloaded candles: 5000 | Last: 2020-01-18 14:05:00
Downloaded candles: 6000 | Last: 2020-01-22 01:25:00
Downloaded candles: 7000 | Last: 2020-01-25 12:45:00
Downloaded candles: 8000 | Last: 2020-01-29 00:05:00
Downloaded candles: 9000 | Last: 2020-02-01 11:25:00
Downloaded candles: 10000 | Last: 2020-02-04 22:45:00
Downloaded candles: 11000 | Last: 2020-02-08 10:05:00
Downloaded candles: 12000 | Last: 2020-02-11 22:25:00
Downloaded candles: 13000 | Last: 2020-02-15 09:45:00
Downloaded candles: 14000 | Last: 2020-02-18 21:05:00
Downloaded candles: 15000 | Last: 2020-02-22 14:15:00
Downloaded candles: 16000 | Last: 2020-02-26 01:35:00
Downloaded candles: 17000 | Last: 2020-02-29 12:55:00
Downloaded candles: 18000 | Last: 2020-03-04 00:15:00
Downloaded candles: 19000 | Last: 202

CELL 7 — Convert to DataFrame


In [7]:
columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

df = pd.DataFrame(all_data, columns=columns)

df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1577836800000,7195.24000000,7196.25000000,7178.64000000,7179.78000000,95.50913300,1577837099999,686317.13625177,1127,32.77324500,235537.29504531,0
1,1577837100000,7179.76000000,7191.77000000,7178.20000000,7191.07000000,59.36522500,1577837399999,426481.26036406,631,24.76651300,177935.61820100,0
2,1577837400000,7193.15000000,7193.53000000,7180.24000000,7180.97000000,48.06851000,1577837699999,345446.50301879,694,19.42228300,139596.62168263,0
3,1577837700000,7180.97000000,7186.40000000,7177.35000000,7178.29000000,32.19292900,1577837999999,231162.55542356,576,12.96325800,93091.43327629,0
4,1577838000000,7177.71000000,7182.46000000,7175.47000000,7176.96000000,49.02739700,1577838299999,351927.89388145,710,22.81974400,163817.88115474,0


CELL 8 — Clean Data

In [8]:
numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col])

df["datetime"] = pd.to_datetime(df["open_time"], unit="ms")

df = df[
    [
        "datetime",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

df.head()

,datetime,open,high,low,close,volume
0,2020-01-01 00:00:00,7195.24,7196.25,7178.64,7179.78,95.509133
1,2020-01-01 00:05:00,7179.76,7191.77,7178.20,7191.07,59.365225
2,2020-01-01 00:10:00,7193.15,7193.53,7180.24,7180.97,48.068510
3,2020-01-01 00:15:00,7180.97,7186.40,7177.35,7178.29,32.192929
4,2020-01-01 00:20:00,7177.71,7182.46,7175.47,7176.96,49.027397


CELL 9 — Save CSV

In [9]:
csv_path = f"{DATA_DIR}/btcusdt_5m_2020_2026.csv"

df.to_csv(csv_path, index=False)

print("CSV Saved:", csv_path)

CSV Saved: ../data/btcusdt_5m_2020_2026.csv


CELL 10 — Save Faster Parquet File

In [10]:
parquet_path = f"{DATA_DIR}/btcusdt_5m_2020_2026.parquet"

df.to_parquet(parquet_path, index=False)

print("Parquet Saved:", parquet_path)

Parquet Saved: ../data/btcusdt_5m_2020_2026.parquet


CELL 11 — Final Dataset Info

In [11]:
print(df.shape)

df.tail()

(630831, 6)


,datetime,open,high,low,close,volume
630826,2025-12-31 23:40:00,87652.46,87686.09,87652.46,87685.31,10.47354
630827,2025-12-31 23:45:00,87685.31,87697.40,87663.92,87690.61,13.07752
630828,2025-12-31 23:50:00,87690.60,87690.61,87641.14,87641.14,13.66626
630829,2025-12-31 23:55:00,87641.15,87655.31,87641.14,87648.22,18.20491
630830,2026-01-01 00:00:00,87648.21,87701.91,87632.74,87701.91,18.44460
